# 05 SQL は Spark でどう実行されるか

**目的**: `sql/` で解いたような少し難しい SQL（ウィンドウ関数、自己結合、NOT IN、相関サブクエリ、集合演算）が、Spark の中でどんな計画になるかを読む（80分）。SQL の書き方の違いが、実行コストの違いとしてどう現れるかをつなげる回。

終わったときに説明できるようになること:

- ウィンドウ関数と自己結合で同じ結果を出したとき、計画とコストがどう違うか
- `NOT IN` と `NOT EXISTS` が、NULL があると結果も計画も変わること
- 相関サブクエリや集合演算が、内部では結合に書き換えられること
- 粒度の違うテーブルを結合する前に集計する、という型を DataFrame API で書けること
- SQL で書いても DataFrame API で書いても、同じ計画になること

先に `sql/problems/self_join.md` と `grain.md` を解いておくと、自分の書いた SQL をそのまま使える。

In [ ]:
from pyspark.sql import functions as F

from labkit import (check_plan, get_spark, large, load_seed_views, materialize, observe, plan_ops,
                    same_optimized_plan, same_rows)

spark = get_spark("05_sql_plans", {
    "spark.sql.adaptive.enabled": "false",
    "spark.sql.autoBroadcastJoinThreshold": "-1",   # 小さいデータでも素の結合方式が見えるように、自動の broadcast を止める
})

load_seed_views(spark)                                       # MySQL と同じ中身のテーブル（customers, orders, ...）
large(spark, "products").createOrReplaceTempView("products_large")   # 速度を比べるための大きい商品テーブル（2万行）

Spark SQL と MySQL の方言の違いで、ここで関係するのは日付の書式くらい（MySQL の `DATE_FORMAT(x, '%Y-%m')` は、Spark では `date_format(x, 'yyyy-MM')`）。

---

## 実験1: 順位付け — ウィンドウ関数と自己結合

`sql/problems/self_join.md` の q3 と同じ問題。カテゴリごとに、定価の高い順に順位を付ける（同額は同順位、次は飛ばす）。

In [ ]:
# TODO 1-1: 同じ順位付けを2通りの SQL で書く。出力する列は category, product_id, list_price, price_rank
#   rank_window  : RANK() OVER (...) を使う
#   rank_selfjoin: ウィンドウ関数を使わず、自己結合と COUNT で書く（sql/answers/self_join/q3.sql に書いたもの）
#   どちらも ORDER BY は付けない（並べ替えのぶんの shuffle を計画に混ぜないため）
rank_window_sql = '''
    ...
'''
rank_selfjoin_sql = '''
    ...
'''
rank_window = spark.sql(rank_window_sql)
rank_selfjoin = spark.sql(rank_selfjoin_sql)

In [ ]:
same_rows(rank_window, rank_selfjoin)

print("ウィンドウ関数:", plan_ops(rank_window))
print("自己結合      :", plan_ops(rank_selfjoin))
rank_selfjoin.explain()

✏️ **観察: それぞれの計画に Exchange はいくつ？ 自己結合の SortMergeJoin の行の末尾には、結合キー以外の条件（list_price の大小）がどう書かれている？**

（ここに書く）

48行では差が出ないので、2万行の `products_large` で同じ SQL を実行して比べる。

In [ ]:
with observe(spark, "ウィンドウ関数（2万行）"):
    materialize(spark.sql(rank_window_sql.replace("FROM products", "FROM products_large")))
with observe(spark, "自己結合（2万行）"):
    materialize(spark.sql(rank_selfjoin_sql.replace("products p", "products_large p")))

✏️ **観察: 時間は何倍違う？ 行数が10倍になったら、それぞれの時間はどう増えると思う？**

（ここに書く）

ウィンドウ関数は、カテゴリごとに1回並べ替えるだけで順位が決まる。自己結合は、同じカテゴリの商品の**全ての組み合わせ**を作ってから数えるので、カテゴリ内の行数の2乗に比例して重くなる。結果が同じでも、どう書くかで計算量が変わる。

---

## 実験2: NOT IN と NULL

「誰のことも紹介していない顧客」を数える。`customers.referrer_id` は「自分を紹介した顧客」で、紹介なしは NULL。

✏️ **予想: 下の2つの書き方は、同じ結果になる？**

（ここに書く）

In [ ]:
# TODO 2-1: 「どの顧客の referrer_id にも現れない顧客」の人数（列名 n）を、2通りの SQL で数える
#   not_in    : WHERE customer_id NOT IN (サブクエリ) を使う
#   not_exists: WHERE NOT EXISTS (相関サブクエリ) を使う
not_in = spark.sql('''
    ...
''')
not_exists = spark.sql('''
    ...
''')

In [ ]:
print("NOT IN    :", not_in.first()["n"], "人")
print("NOT EXISTS:", not_exists.first()["n"], "人")

print("\n=== NOT IN の計画 ===")
not_in.explain()
print("=== NOT EXISTS の計画 ===")
not_exists.explain()

✏️ **観察: 結果はどう違った？ なぜ NOT IN はその人数になる？ 2つの計画で、結合の種類（LeftAnti）は同じだが、結合方式は違う。それぞれ何？**

（ここに書く）

サブクエリの結果に NULL が1つでもあると、`x NOT IN (..., NULL)` は決して TRUE にならない（`x <> NULL` が UNKNOWN になるため）。だから `NOT IN` は 0 人を返す。MySQL でも同じ結果になる。

Spark は、この NULL の特別扱いを正しく実行するために、`NOT IN` を専用の結合（null-aware anti join）にする。計画の `BroadcastHashJoin [...], LeftAnti, BuildRight, true` の最後の `true` がその印で、自動の broadcast を止めていても、サブクエリ側を必ず broadcast する。サブクエリが大きいと、メモリ不足の原因になる。

`NOT EXISTS` は NULL を気にしなくて良いので、普通の LeftAnti 結合になり、大きいテーブル同士でも SortMergeJoin で動く。**「〜にないもの」を探すときは `NOT EXISTS`（DataFrame API なら `left_anti` の join）で書く**、と覚えておけば、結果の間違いと性能の問題を両方避けられる。

---

## 実験3: 相関サブクエリは結合に書き換えられる

各顧客の「最後の completed 注文の日時」を出す。

In [ ]:
# TODO 3-1: SELECT 句の中に相関サブクエリを書いて、各顧客の最後の completed 注文日時を出す
#   出力する列は customer_id, last_ordered_at（注文のない顧客は NULL）
by_subquery = spark.sql('''
    ...
''')

In [ ]:
by_subquery.explain()
check_plan(by_subquery, has=["LeftOuter"], lacks=["Subquery"])

✏️ **観察: 「顧客1人ごとにサブクエリを1回実行する」ような計画になっている？ サブクエリは、どんな演算子の組み合わせに変わった？**

（ここに書く）

In [ ]:
# TODO 3-2: 同じものを DataFrame API で書く。計画から読み取った形のとおり、「先に集計して、外部結合する」
#   使うもの: spark.table("orders") / spark.table("customers") / filter / groupBy / agg / join(..., "customer_id", "left")
by_dataframe = ...

In [ ]:
same_rows(by_subquery, by_dataframe)
print("相関サブクエリ:", plan_ops(by_subquery))
print("DataFrame API :", plan_ops(by_dataframe))

Spark は相関サブクエリを見つけると、「サブクエリ側を相関キーで集計したもの」との外部結合に書き換える（decorrelation）。MySQL が相関サブクエリを行ごとに実行することがあるのとは対照的で、Spark では「相関サブクエリだから遅い」とは限らない。ただし、書き換えられない形のサブクエリはエラーになる。

---

## 実験4: 集合演算も結合になる

In [ ]:
# TODO 4-1: 2025年にも2026年にも completed 注文がある顧客の customer_id を、INTERSECT で出す
both_years = spark.sql('''
    ...
''')

In [ ]:
print(f"{both_years.count():,} 人")
both_years.explain()
check_plan(both_years, has=["LeftSemi"])

✏️ **観察: INTERSECT はどの種類の結合になった？ 結合キーが customer_id そのものではなく、coalesce(customer_id, 0) と isnull(customer_id) の2つになっているのはなぜ？（集合演算での NULL の扱いを思い出す）**

（ここに書く）

`INTERSECT` は LeftSemi 結合（＋重複排除）、`EXCEPT` は LeftAnti 結合になる。普通の結合の `=` は NULL 同士を一致させないが、集合演算は NULL 同士を同じ値として扱う。その違いを吸収するために、Spark は結合キーを「NULL なら 0 に置き換えた値」と「NULL かどうか」の組にしている。`sql/problems/set_operations.md` の q3 で、差分の検証に `EXCEPT` や `<=>` を使ったのと同じ話だ。

---

## 実験5: 粒度をそろえてから結合する

`sql/problems/grain.md` の q1 と同じ問題。注文・明細・支払を素直に結合すると、金額が膨らむ。

In [ ]:
inflated = spark.sql('''
    SELECT date_format(o.ordered_at, 'yyyy-MM') AS ym,
           SUM(i.quantity * i.unit_price) AS item_amount,
           SUM(p.amount)                  AS paid_amount
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    JOIN payments p    ON p.order_id = o.order_id
    WHERE o.status = 'completed' AND o.ordered_at >= '2026-01-01' AND o.ordered_at < '2026-11-01'
    GROUP BY 1
    ORDER BY 1
''')
inflated.show(3)

n_orders = spark.sql("SELECT COUNT(*) FROM orders WHERE status = 'completed'").first()[0]
n_joined = spark.sql('''
    SELECT COUNT(*) FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    JOIN payments p    ON p.order_id = o.order_id
    WHERE o.status = 'completed'
''').first()[0]
print(f"completed 注文 {n_orders:,} 件 → 3表を結合すると {n_joined:,} 行")

In [ ]:
# TODO 5-1: DataFrame API で正しい月次の数字を出す。明細と支払を、結合する前にそれぞれ「1注文1行」に集計する
#   出力する列は ym（"yyyy-MM"）, item_amount, paid_amount。対象は completed で 2026-01-01 以上 2026-11-01 未満
orders_df = spark.table("orders")
item_total = ...     # order_id ごとの明細金額（quantity * unit_price の合計）
pay_total = ...      # order_id ごとの支払金額（amount の合計）
monthly = ...

In [ ]:
result = monthly.orderBy("ym").collect()
for r in result:
    print(r["ym"], f"{r['item_amount']:>12,} {r['paid_amount']:>12,}")

ok = len(result) == 10 and all(r["item_amount"] == r["paid_amount"] for r in result)
print("✅ 10か月ぶんあり、どの月も明細合計と支払合計が一致" if ok else "❌ 月の数が違うか、明細合計と支払合計が一致しない月がある")

このデータは、completed 注文の支払合計が明細合計と一致するように作ってある。2つの列が一致していれば、どちらも水増しされていない。

結合で行が増えるかどうかは、**結合キーが相手のテーブルで一意かどうか**で決まる。`order_id` は orders では一意だが、order_items と payments では一意でない。結合する前に「このキーは相手側で一意か」を確かめる習慣が、いちばん安い予防策になる。

---

## 実験6: SQL と DataFrame API は同じ計画になる

カテゴリ別の completed 売上を、SQL と DataFrame API の両方で書く。

In [ ]:
# TODO 6-1: カテゴリ別の completed 売上（列名 category, amount）を、SQL と DataFrame API で書く
#   結合の順番は orders → order_items → products、結合キーは order_id と product_id
by_sql = spark.sql('''
    ...
''')
by_api = ...

In [ ]:
by_sql.orderBy("category").show()
same_rows(by_sql, by_api)
same_optimized_plan(by_sql, by_api)
print("SQL          :", plan_ops(by_sql))
print("DataFrame API:", plan_ops(by_api))

SQL も DataFrame API も、同じ論理計画に変換されてから最適化される。どちらで書いても性能は同じなので、読みやすいほうを選べば良い。

最適化後の計画が「違う」と出ても、演算子の並びが同じなら、列の順番や別名が違うだけのことが多い。`explain()` で見比べて、どこが違うかを確かめる。

---

## まとめ

✏️ **このノートブックで分かったことを、自分の言葉で3行にまとめる**

（ここに書く）

### 説明できるか確かめる

1. 同じ結果を出すウィンドウ関数と自己結合で、計算量が違うのはなぜ？
2. `NOT IN (サブクエリ)` を避けるべき理由を、結果と性能の両面から説明できる？
3. Spark は相関サブクエリをどう実行する？
4. `INTERSECT` / `EXCEPT` と、結合の `=` では、NULL の扱いがどう違う？
5. 結合で行数が増えてしまうのは、どんなとき？ 結合する前に何を確かめる？
6. 「SQL で書くのと DataFrame API で書くのと、どちらが速い？」と聞かれたら、どう答える？

In [ ]:
spark.stop()